# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
I'm picking Refresh / Content Opportunity Scoring. The starter dataset is built for exactly
this: one row per content item with trailing-90-day engagement, CTR, and position metrics,
plus a pre-computed decline label I have to treat carefully rather than trust. Unlike Ranking
Signal Analysis or Clustering, this lane produces something an editor can act on directly — a
ranked queue, not just a finding. I can start entirely on the 30k-row starter CSV and only
reach for the warehouse if I need more history per client.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
Decision: which content items should an editor review for a refresh this week, out of
everything they could review?

Unit of analysis: one content item (pseudonymized content_id), scored using its trailing-90-day
metrics.

Who acts, and how: a content editor with limited hours works down a ranked queue, starting at
the top.

Output: a ranked review queue — each item gets a score and a reason code (e.g. "CTR below peers
at this position," "scroll_rate falling," "AI-traffic dependent and declining").

Cost of a wrong call: a false positive (flagged but fine) wastes editor hours — the cheaper
mistake. A false negative (real decline missed) lets a page keep losing traffic silently until
the next review cycle — the more expensive mistake. Because these costs are asymmetric, I care
more about precision at the top of the queue than overall accuracy.

Why not a plain rule: a single-threshold rule (e.g. "if ctr < X, flag it") ignores that
content_type drives missingness and that scroll_rate/ai_traffic_pct aren't on a normal 0-100
scale. A hand-written rule either false-flags whole content types or ignores the AI-traffic
signal entirely. The pattern is real but too tangled across ~44 columns to hand-code.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
Real number 1: how much position data is actually missing. avg_position = 0 is a sentinel for
"no data," not an actual rank of zero — I need to know how much of the dataset this affects
before scoring on position.


SyntaxError: invalid character '—' (U+2014) (586037184.py, line 2)

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

no_position = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 (no data, not rank 0): {no_position} / {len(df)}")
print(f"That's {no_position / len(df):.1%} of all rows")

Real number 2: missingness isn't random — it follows content_type. A blind fillna(0) would
inject a fake signal for whichever content_type has high missingness, so I need has_-flags
instead.

In [ ]:
missing_by_type = df.groupby("content_type")["keyword_count"].apply(lambda s: s.isna().mean())
print("Missing keyword_count rate by content_type:")
print(missing_by_type.sort_values(ascending=False))

Real number 3: scroll_rate and ai_traffic_pct aren't normal percentages — they can exceed 100
because the numerator and denominator come from different measurement systems. Treating them
like ordinary rates would misrank items with high but legitimate values.

In [ ]:
over_100 = ((df["scroll_rate"] > 100) | (df["ai_traffic_pct"] > 100)).sum()
print(f"Rows where scroll_rate or ai_traffic_pct > 100: {over_100} / {len(df)}")
print(f"That's {over_100 / len(df):.1%} of all rows")

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

I can claim directional, decision-support results: "this item scores high for review priority,"
not "this item will decline." I can't use is_declining_label or trend_direction as a feature —
both are derived from trend_pct, so using them would let the model learn a rule I already wrote,
not a real pattern. I can't claim causal language ("refreshing this page will recover traffic")
— the dataset shows association, not intervention effects. I can't claim the queue generalizes
to high-missingness content types unless has_-flags were used instead of a blind fillna(0). Any
precision number I report needs a naive baseline (e.g. rank by avg_position alone) to be
meaningful, since a high-looking precision could just be the base rate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.